In [1]:
import os
import random
import pickle
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy.io
from scipy.io import loadmat

import mne
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    confusion_matrix,
)

from tensorflow.keras.constraints import max_norm
from tensorflow.keras.layers import (
    Activation,
    AveragePooling2D,
    BatchNormalization,
    Conv2D,
    Dense,
    DepthwiseConv2D,
    Dropout,
    Flatten,
    Input,
    Reshape,
    SeparableConv2D,
    SpatialDropout2D,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os
from typing import Tuple

import tensorflow as tf
from tensorflow.keras import Input, Model, layers, models

import numpy as np
from scipy.signal import welch, butter, filtfilt
import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras import layers, models

2026-04-30 15:12:37.454237: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777561957.712612      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777561957.785918      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777561958.372967      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777561958.373012      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777561958.373015      23 computation_placer.cc:177] computation placer alr

In [2]:
class EEGDataset_ADHD_TF:
    """
    Subject-level dataset (one .mat file = one subject)
    Each item: (fname, eeg_epochs_np, label_int)
      eeg_epochs_np: (E, C, T)
    """

    def __init__(self, adhd_dir, control_dir,
                 lowcut=0.5, highcut=60.0, notch=50.0,
                 window=2.0, overlap=0.5,
                 default_fs=128):

        self.samples = []
        self.lowcut = float(lowcut)
        self.highcut = float(highcut)
        self.notch = float(notch)
        self.window = float(window)
        self.overlap = float(overlap)
        self.default_fs = int(default_fs)

        self._process_folder(adhd_dir, label=1)
        self._process_folder(control_dir, label=0)

        if len(self.samples) == 0:
            raise ValueError("No subjects loaded. Check folders and file contents.")

    def _process_folder(self, folder, label):
        if not os.path.isdir(folder):
            raise ValueError(f"Directory does not exist: {folder}")

        for fname in sorted(os.listdir(folder)):
            if not fname.lower().endswith(".mat"):
                continue
            mat_path = os.path.join(folder, fname)
            eeg = self._process_mat(mat_path)
            if eeg is not None:
                self.samples.append((fname, eeg.astype(np.float32), int(label)))

    def _process_mat(self, file_path):
        mat = loadmat(file_path)

        # Variable name is usually the same as filename (e.g., v10p.mat -> "v10p")
        key = os.path.splitext(os.path.basename(file_path))[0]
        if key not in mat:
            # fallback: first 2D array
            key = None
            for k, v in mat.items():
                if isinstance(v, np.ndarray) and v.ndim == 2 and v.size > 0:
                    key = k
                    break
            if key is None:
                return None

        data = np.asarray(mat[key], dtype=np.float64)  # often (T, C)

        # Find fs if present
        fs = None
        for k in mat.keys():
            kl = k.lower()
            if ("fs" in kl) or ("freq" in kl) or ("sampling" in kl) or ("sfreq" in kl):
                try:
                    fs = int(np.squeeze(mat[k]))
                    break
                except Exception:
                    fs = None

        if fs is None or fs <= 0:
            fs = self.default_fs

        # Ensure (C, T) for MNE
        if data.ndim != 2:
            return None

        if data.shape[1] <= 256 and data.shape[0] > data.shape[1]:
            data = data.T  # (C, T)

        n_ch, n_times = data.shape

        min_samples = int(np.ceil(self.window * fs))
        if n_times < min_samples:
            return None

        ch_names = [f"Ch{i+1}" for i in range(n_ch)]
        info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types=["eeg"] * n_ch)
        raw = mne.io.RawArray(data, info, verbose=False)

        raw.set_eeg_reference("average", verbose=False)

        # Evita errores si notch o highcut quedan por encima de Nyquist
        nyq = fs / 2.0

        if self.notch < nyq:
            raw.notch_filter(freqs=[self.notch], method="iir", verbose=False)

        highcut = min(self.highcut, nyq - 1.0)

        raw.filter(self.lowcut, highcut, method="iir", verbose=False)

        step = self.window * (1.0 - self.overlap)
        if step <= 0:
            raise ValueError("overlap must be < 1.0 so that step > 0")

        epochs = mne.make_fixed_length_epochs(
            raw,
            duration=self.window,
            overlap=self.window - step,
            preload=True,
            verbose=False
        )

        eeg_data = epochs.get_data()  # (E, C, T)
        if eeg_data.size == 0:
            return None

        return eeg_data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [3]:
def cnn_lstm_eegnet(
        *,
        n_channels: int = 56,
        n_times: int = 385,
        n_classes: int = 3,
        temporal_filters: int = 50,
        temporal_kernel: int = 50,     # samples → 200 ms @ 256 Hz
        depth_multiplier: int = 2,
        pool_size: int = 40,
        pool_stride: int = 20,
        dropout_rate: float = 0.5,
        lstm_units_1: int = 10,        # paper uses 10 + 10 = 20 total
        lstm_units_2: int = 10,
        activation: str = "elu",
        use_bias_conv: bool = False,
        name: str = "cnn_lstm_eegnet"
    ) -> tf.keras.Model:
    """
    Return an **uncompiled** CNN-LSTM model replicating Wang et al. (2022).

    Change the keyword arguments to vary the architecture.
    """
    act = layers.Activation(activation)

    # ── 1. Input ────────────────────────────────────────────────────
    inp = layers.Input(shape=(n_channels, n_times, 1))

    # ── 2a. Temporal convolution ───────────────────────────────────
    x = layers.Conv2D(filters=temporal_filters,
                      kernel_size=(1, temporal_kernel),
                      padding="same",
                      use_bias=use_bias_conv)(inp)
    x = layers.BatchNormalization()(x)
    x = act(x)

    # ── 2b. Depth-wise spatial convolution ─────────────────────────
    x = layers.DepthwiseConv2D(kernel_size=(n_channels, 1),
                               depth_multiplier=depth_multiplier,
                               use_bias=use_bias_conv)(x)
    x = layers.BatchNormalization()(x)
    x = act(x)

    # ── 2c. Pool + dropout ─────────────────────────────────────────
    x = layers.AveragePooling2D(pool_size=(1, pool_size),
                                strides=(1, pool_stride))(x)
    x = layers.Dropout(dropout_rate)(x)

    # ── 3. LSTM block ──────────────────────────────────────────────
    x = layers.Permute((2, 1, 3))(x)                 # (batch, time, 1, feat)
    x = layers.TimeDistributed(layers.Flatten())(x)  # (batch, time, feat)

    x = layers.LSTM(lstm_units_1, return_sequences=True)(x)
    x = layers.LSTM(lstm_units_2)(x)

    # ── 4. Classifier ──────────────────────────────────────────────
    out = layers.Dense(n_classes, activation="softmax")(x)

    return models.Model(inp, out, name=name)

In [4]:
# =========================================================
# BUILD EPOCH ARRAYS + DATASETS + CALLBACKS + PLOTS
# Compatible with cnn_lstm_eegnet
# Input model: (N, C, T, 1)
# Output model: softmax -> (N, n_classes)
# Labels: sparse integer labels -> 0=Control, 1=ADHD
# =========================================================

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay


# =========================================================
# BUILD EPOCH-LEVEL ARRAYS FROM SUBJECT-LEVEL DATASET
# =========================================================

def build_epoch_arrays(dataset_obj):
    """
    Converts subject-level dataset into epoch-level arrays.

    Input dataset item:
        name, eeg_epochs, label

    where:
        eeg_epochs: (E, C, T)
        label: int

    Returns:
        X:      (N, C, T, 1)
        y:      (N,)
        groups: (N,) subject-name per epoch
    """

    X_list, y_list, g_list = [], [], []

    for i in range(len(dataset_obj)):
        name, eeg_epochs, label = dataset_obj[i]  # (E, C, T)
        E = int(eeg_epochs.shape[0])

        X_list.append(eeg_epochs)
        y_list.append(np.full((E,), int(label), dtype=np.int32))
        g_list.append(np.full((E,), str(name), dtype=object))

    X = np.concatenate(X_list, axis=0).astype(np.float32)   # (N, C, T)
    y = np.concatenate(y_list, axis=0).astype(np.int32)     # (N,)
    groups = np.concatenate(g_list, axis=0)                 # (N,)

    # Important for cnn_lstm_eegnet:
    # model input is (n_channels, n_times, 1)
    X = X[..., np.newaxis].astype(np.float32)               # (N, C, T, 1)

    return X, y, groups


# =========================================================
# SUBJECT-LEVEL LABELS DERIVED FROM EPOCH-LEVEL GROUPS/Y
# =========================================================

def build_subject_table_from_epochs(groups_epoch, y_epoch):
    """
    Builds subject-level arrays from epoch-level labels.

    Returns:
        subj_names:  (S,)
        subj_labels: (S,)
    """

    subj_names = np.unique(groups_epoch.astype(str))
    subj_labels = []

    for s in subj_names:
        mask = groups_epoch.astype(str) == s
        ys = np.unique(y_epoch[mask])

        if len(ys) != 1:
            raise ValueError(f"Subject {s} has multiple labels across epochs: {ys}")

        subj_labels.append(int(ys[0]))

    return subj_names, np.array(subj_labels, dtype=np.int32)


# =========================================================
# GET EPOCH INDICES FROM SUBJECT NAMES
# =========================================================

def epoch_indices_from_subjects(groups_epoch, subject_names_subset):
    """
    Given a list/array of subject names, returns the epoch indices
    belonging to those subjects.
    """

    subject_names_subset = np.array(subject_names_subset).astype(str)

    return np.where(
        np.isin(groups_epoch.astype(str), subject_names_subset)
    )[0]


# =========================================================
# TF.DATA BUILDERS
# =========================================================

def make_ds_from_indices(
    X,
    y,
    groups,
    idxs,
    training,
    with_groups,
    batch_size,
    seed
):
    """
    Creates a tf.data.Dataset from selected epoch indices.

    If with_groups=True:
        returns (x, y, group)
    else:
        returns (x, y)
    """

    x = X[idxs].astype(np.float32)      # (B, C, T, 1)
    yy = y[idxs].astype(np.int32)       # (B,)

    if with_groups:
        gg = groups[idxs].astype(str)
        ds = tf.data.Dataset.from_tensor_slices((x, yy, gg))
    else:
        ds = tf.data.Dataset.from_tensor_slices((x, yy))

    if training:
        ds = ds.shuffle(
            buffer_size=len(idxs),
            seed=seed,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


# =========================================================
# VALIDATION BALANCED ACCURACY CALLBACK
# Compatible with softmax output
# =========================================================

class ValBalancedAccuracy(tf.keras.callbacks.Callback):
    """
    Computes validation balanced accuracy at the end of each epoch.

    This version assumes:
        model output: softmax probabilities with shape (B, n_classes)
        y labels: sparse integer labels with shape (B,)
    """

    def __init__(self, val_ds_xy):
        super().__init__()
        self.val_ds = val_ds_xy
        self.best = -np.inf
        self.last = None

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []

        for xb, yb in self.val_ds:
            prob = self.model(xb, training=False).numpy()   # (B, n_classes)
            pred = np.argmax(prob, axis=1).astype(int)

            y_true.extend(yb.numpy().astype(int).tolist())
            y_pred.extend(pred.tolist())

        bacc = balanced_accuracy_score(y_true, y_pred)

        self.last = float(bacc)
        self.best = max(self.best, self.last)

        if logs is not None:
            logs["val_balanced_accuracy"] = self.last

        print(f" - val_balanced_accuracy: {self.last:.4f}", end="")


# =========================================================
# PLOT TRAINING CURVE FOR ONE FOLD
# =========================================================

def plot_fold_training_curve(
    history,
    fold_id,
    save_path,
    metrics=("loss", "acc", "val_balanced_accuracy")
):
    """
    Saves training curves for one fold.
    """

    n_metrics = len(metrics)

    fig, axes = plt.subplots(
        1,
        n_metrics,
        figsize=(5 * n_metrics, 4)
    )

    if n_metrics == 1:
        axes = [axes]

    fig.suptitle(
        f"Fold {fold_id + 1} — Training Curves",
        fontsize=13
    )

    epochs_range = range(1, len(history["loss"]) + 1)

    for col, m in enumerate(metrics):
        ax = axes[col]

        if m in history:
            ax.plot(
                epochs_range,
                history[m],
                label=m,
                linewidth=1.2
            )

        val_key = f"val_{m}"

        if val_key in history:
            ax.plot(
                epochs_range,
                history[val_key],
                label=val_key,
                linewidth=1.2,
                linestyle="--"
            )

        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {save_path}")


# =========================================================
# PLOT TRAINING CURVES FOR ALL FOLDS
# =========================================================

def plot_all_training_curves(
    fold_histories,
    save_dir,
    metrics=("loss", "acc", "val_balanced_accuracy")
):
    """
    Saves:
        1. Grid with curves per fold.
        2. Overlay plot with all folds.
    """

    n_folds = len(fold_histories)
    n_metrics = len(metrics)

    fig, axes = plt.subplots(
        n_folds,
        n_metrics,
        figsize=(5 * n_metrics, 3.5 * n_folds),
        squeeze=False
    )

    fig.suptitle(
        "Training Curves per Fold",
        fontsize=14,
        y=1.01
    )

    for row, fid in enumerate(sorted(fold_histories.keys())):
        hist = fold_histories[fid]
        epochs_range = range(1, len(hist["loss"]) + 1)

        for col, m in enumerate(metrics):
            ax = axes[row, col]

            if m in hist:
                ax.plot(
                    epochs_range,
                    hist[m],
                    label=m,
                    linewidth=1.0
                )

            val_key = f"val_{m}"

            if val_key in hist:
                ax.plot(
                    epochs_range,
                    hist[val_key],
                    label=val_key,
                    linewidth=1.0,
                    linestyle="--"
                )

            ax.set_title(f"Fold {fid + 1} — {m}", fontsize=9)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(m)
            ax.legend(fontsize=6)
            ax.grid(True, alpha=0.3)

    plt.tight_layout()

    path = os.path.join(save_dir, "training_curves_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {path}")

    # Overlay
    fig2, axes2 = plt.subplots(
        1,
        n_metrics,
        figsize=(5 * n_metrics, 4)
    )

    if n_metrics == 1:
        axes2 = [axes2]

    fig2.suptitle("Training Curves Overlay — All Folds", fontsize=14)

    for col, m in enumerate(metrics):
        ax = axes2[col]

        for fid in sorted(fold_histories.keys()):
            hist = fold_histories[fid]
            epochs_range = range(1, len(hist["loss"]) + 1)

            if m in hist:
                ax.plot(
                    epochs_range,
                    hist[m],
                    label=f"Fold {fid + 1}",
                    linewidth=1.0,
                    alpha=0.7
                )

        ax.set_title(f"All Folds — {m}", fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()

    path2 = os.path.join(save_dir, "training_curves_overlay.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)

    print(f"Saved: {path2}")


# =========================================================
# PLOT CONFUSION MATRIX FOR ONE FOLD
# =========================================================

def plot_fold_confusion_matrix(
    cm,
    fold_id,
    save_path,
    class_names=("Control", "ADHD")
):
    """
    Saves one confusion matrix.
    """

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))

    disp = ConfusionMatrixDisplay(
        cm,
        display_labels=class_names
    )

    disp.plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
        values_format="d"
    )

    ax.set_title(
        f"Fold {fold_id + 1} — Confusion Matrix",
        fontsize=11
    )

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {save_path}")


# =========================================================
# PLOT CONFUSION MATRICES FOR ALL FOLDS
# =========================================================

def plot_all_confusion_matrices(
    fold_cms,
    super_cm,
    save_dir,
    class_names=("Control", "ADHD")
):
    """
    Saves:
        1. Confusion matrices per fold.
        2. Accumulated confusion matrix.
    """

    n_folds = len(fold_cms)
    ncols = min(n_folds, 5)
    nrows = int(np.ceil(n_folds / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4 * ncols, 4 * nrows),
        squeeze=False
    )

    fig.suptitle(
        "Confusion Matrices per Fold",
        fontsize=14,
        y=1.01
    )

    for i, fid in enumerate(sorted(fold_cms.keys())):
        row, col = divmod(i, ncols)
        ax = axes[row][col]

        disp = ConfusionMatrixDisplay(
            fold_cms[fid],
            display_labels=class_names
        )

        disp.plot(
            ax=ax,
            cmap="Blues",
            colorbar=False,
            values_format="d"
        )

        ax.set_title(f"Fold {fid + 1}", fontsize=10)

    for j in range(i + 1, nrows * ncols):
        row, col = divmod(j, ncols)
        axes[row][col].axis("off")

    plt.tight_layout()

    path = os.path.join(save_dir, "confusion_matrices_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {path}")

    fig2, ax2 = plt.subplots(1, 1, figsize=(5, 5))

    disp2 = ConfusionMatrixDisplay(
        super_cm,
        display_labels=class_names
    )

    disp2.plot(
        ax=ax2,
        cmap="Oranges",
        colorbar=False,
        values_format="d"
    )

    ax2.set_title(
        "Accumulated Confusion Matrix — All Folds",
        fontsize=12
    )

    plt.tight_layout()

    path2 = os.path.join(save_dir, "confusion_matrix_accumulated.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)

    print(f"Saved: {path2}")

In [5]:
def segmentar_senales(db, labels):
    """
    Divide las señales EEG en segmentos de 512 instantes con traslape del 50%.

    Compatible con cnn_lstm_eegnet:
        X.shape -> (N, C, T, 1)

    Args:
        db (dict):
            Diccionario donde las claves son los nombres de los sujetos y los valores
            son matrices de forma (C, T_i).

        labels:
            Puede ser:
                - dict: labels[sujeto] -> etiqueta
                - array/list: etiquetas en el mismo orden de db.items()

    Returns:
        segmentos:
            np.ndarray con forma (N, C, 512, 1)

        y:
            np.ndarray con forma (N,)

        sbjs:
            np.ndarray con el sujeto asociado a cada segmento

        window_ids:
            np.ndarray con el identificador de ventana por segmento
    """

    segmento_tamano = 512
    paso = int(segmento_tamano * 0.5)  # 50% overlap

    segmentos = []
    y = []
    sbjs = []
    window_ids = []

    for i, (sujeto, senal) in enumerate(db.items()):

        senal = np.asarray(senal, dtype=np.float32)

        if senal.ndim != 2:
            raise ValueError(
                f"La señal del sujeto {sujeto} debe tener forma (C, T), "
                f"pero tiene forma {senal.shape}."
            )

        C, T = senal.shape

        if T < segmento_tamano:
            print(
                f"Advertencia: sujeto {sujeto} omitido porque T={T} "
                f"es menor que segmento_tamano={segmento_tamano}."
            )
            continue

        # Soporta labels como diccionario o como lista/array
        if isinstance(labels, dict):
            label_sujeto = int(labels[sujeto])
        else:
            label_sujeto = int(labels[i])

        window_count = 1

        for inicio in range(0, T - segmento_tamano + 1, paso):
            segmento = senal[:, inicio:inicio + segmento_tamano]  # (C, 512)

            segmentos.append(segmento)
            y.append(label_sujeto)
            sbjs.append(str(sujeto))
            window_ids.append(f"Window {window_count}")

            window_count += 1

    segmentos = np.asarray(segmentos, dtype=np.float32)  # (N, C, 512)
    y = np.asarray(y, dtype=np.int32)                    # (N,)
    sbjs = np.asarray(sbjs, dtype=object)                # (N,)
    window_ids = np.asarray(window_ids, dtype=object)    # (N,)

    # =====================================================
    # Importante para cnn_lstm_eegnet:
    # pasa de (N, C, T) a (N, C, T, 1)
    # =====================================================
    segmentos = segmentos[..., np.newaxis]               # (N, C, 512, 1)

    return segmentos, y, sbjs, window_ids

In [6]:
# ============================================================
# CNN-LSTM EEGNET FIJO: 5 folds fijos + 10 semillas
# 10 seeds x 5 folds = 50 corridas limpias
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from collections import defaultdict
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from tensorflow.keras.optimizers import Adam


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

MODEL_NAME = "cnn_lstm_eegnet"

SEED_FOLDS = 123
TRAINING_SEEDS = [42, 123, 456, 789, 2024, 7, 99, 314, 2718, 5000]

N_FOLDS = 5
BATCH_SIZE = 64
EPOCHS_FINAL = 100

EXCLUDED_SUBJECTS = {"v56p"}

DATA_ROOT = "/kaggle/input/datasets/javierceron/tdha-data"

RESULTS_ROOT = "/kaggle/working/results_cnn_lstm_eegnet_10seeds"

PARTITIONS_DIR = os.path.join(RESULTS_ROOT, "partitions")

for d in [RESULTS_ROOT, PARTITIONS_DIR]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# SEED HELPER
# ============================================================

def set_all_seeds(seed, deterministic=True):
    tf.keras.backend.clear_session()
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    if deterministic:
        os.environ["TF_DETERMINISTIC_OPS"] = "1"
        try:
            tf.config.experimental.enable_op_determinism()
        except Exception:
            pass


# ============================================================
# HPs FIJOS CNN-LSTM EEGNET
# ============================================================

CNN_LSTM_FIXED_MODEL_ARGS = {
    "n_classes": 2,
    "temporal_filters": 50,
    "temporal_kernel": 25,
    "depth_multiplier": 2,
    "pool_size": 20,
    "pool_stride": 10,
    "dropout_rate": 0.5,
    "lstm_units_1": 10,
    "lstm_units_2": 10,
    "activation": "elu",
    "use_bias_conv": False,
    "name": "cnn_lstm_eegnet",
}

CNN_LSTM_FIXED_COMPILE_ARGS = {
    "learning_rate": 1e-2,
}

# ============================================================
# DATASET BUILDER CNN-LSTM EEGNET
# Modelo:
#   entrada -> (N, C, T, 1)
#   salida  -> softmax (N, 2)
#   labels  -> enteras 0/1
# ============================================================

def make_cnn_lstm_ds_from_indices(X, y, idxs, training, batch_size, seed):
    """
    Dataset para entrenamiento de CNN-LSTM EEGNet.

    Entrada:
        X: (N, C, T) o (N, C, T, 1)

    Salida:
        x: (N, C, T, 1)
        y: entero sparse (N,)
    """

    x = X[idxs].astype(np.float32)

    if x.ndim == 3:
        x = x[..., np.newaxis]  # (N, C, T, 1)

    yy = y[idxs].astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((x, yy))

    if training:
        ds = ds.shuffle(
            buffer_size=len(idxs),
            seed=seed,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


def make_cnn_lstm_ds_from_indices_with_groups(X, y, groups, idxs, training, batch_size, seed):
    """
    Dataset para evaluación a nivel de sujeto.

    Entrada:
        X: (N, C, T) o (N, C, T, 1)

    Salida:
        x: (N, C, T, 1)
        y: entero
        group: nombre del sujeto
    """

    x = X[idxs].astype(np.float32)

    if x.ndim == 3:
        x = x[..., np.newaxis]  # (N, C, T, 1)

    yy = y[idxs].astype(np.int32)
    gg = groups[idxs].astype(str)

    ds = tf.data.Dataset.from_tensor_slices((x, yy, gg))

    if training:
        ds = ds.shuffle(
            buffer_size=len(idxs),
            seed=seed,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


# ============================================================
# EVALUACIÓN A NIVEL DE SUJETO PARA SOFTMAX
# ============================================================

def evaluate_subject_level_softmax(model, test_ds_xyg, threshold=0.5):
    """
    Evaluación por sujeto.

    Para cada época:
        prob_ADHD = softmax[:, 1]
        pred_epoch = prob_ADHD >= threshold

    Para cada sujeto:
        y_pred_subject = votación mayoritaria de las predicciones por época
        y_prob_subject = promedio de prob_ADHD por sujeto
    """

    subject_preds = defaultdict(list)
    subject_probs = defaultdict(list)
    subject_true = {}

    for xb, yb, sb in test_ds_xyg:
        raw_pred = model(xb, training=False).numpy()

        if raw_pred.ndim != 2 or raw_pred.shape[1] != 2:
            raise ValueError(
                f"Se esperaba salida softmax (B, 2), pero llegó {raw_pred.shape}"
            )

        # Probabilidad de la clase positiva: ADHD/TDAH = 1
        prob = raw_pred[:, 1].astype(np.float32)
        pred = (prob >= threshold).astype(int)

        y_np = yb.numpy().astype(int)
        s_np = sb.numpy()

        for name, p, pr, yt in zip(s_np, pred, prob, y_np):
            name = name.decode("utf-8") if isinstance(name, bytes) else str(name)

            subject_preds[name].append(int(p))
            subject_probs[name].append(float(pr))
            subject_true[name] = int(yt)

    y_true, y_pred, y_prob = [], [], []

    for subj in sorted(subject_preds.keys()):
        y_true.append(subject_true[subj])

        # Votación mayoritaria de las épocas del sujeto
        y_pred_subject = int(np.bincount(subject_preds[subj]).argmax())
        y_pred.append(y_pred_subject)

        # Probabilidad promedio del sujeto
        y_prob_subject = float(np.mean(subject_probs[subj]))
        y_prob.append(y_prob_subject)

    # Métricas a nivel de sujeto
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    if len(np.unique(y_true)) < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(y_true, y_prob)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return {
        "accuracy": float(acc),
        "balanced_acc": float(bacc),
        "recall": float(recall),
        "precision": float(precision),
        "f1": float(f1),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "cm": cm,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


# ============================================================
# DATASET
# ============================================================

adhd_dir = os.path.join(DATA_ROOT, "ADHD")
control_dir = os.path.join(DATA_ROOT, "Control")

dataset_tf = EEGDataset_ADHD_TF(
    adhd_dir=adhd_dir,
    control_dir=control_dir,
    lowcut=0.5,
    highcut=60.0,
    notch=50.0,
    window=2.0,
    overlap=0.5,
    default_fs=128
)

n_before = len(dataset_tf.samples)

dataset_tf.samples = [
    sample for sample in dataset_tf.samples
    if os.path.splitext(sample[0])[0] not in EXCLUDED_SUBJECTS
]

n_after = len(dataset_tf.samples)

print(f"Excluded subjects: {sorted(EXCLUDED_SUBJECTS)}")
print(f"Subjects removed: {n_before - n_after}")
print(f"Remaining subjects: {n_after}")

if len(dataset_tf.samples) == 0:
    raise ValueError("No subjects left after exclusion.")

X, y, groups = build_epoch_arrays(dataset_tf)

# Si build_epoch_arrays ya devuelve (N, C, T, 1), se conserva.
# Si devuelve (N, C, T), los dataset builders agregan el canal.
N_CH, N_T = int(X.shape[1]), int(X.shape[2])

print("Epoch arrays:")
print("  X:", X.shape, X.dtype)
print("  y:", y.shape, "labels:", np.unique(y))
print("  groups:", groups.shape, "n_subjects:", len(np.unique(groups.astype(str))))
print("Model input shape (C, T, 1):", (N_CH, N_T, 1))

subj_names, subj_labels = build_subject_table_from_epochs(groups, y)

print("Subject table:")
print("  n_subjects:", len(subj_names))
print("  Control:", int(np.sum(subj_labels == 0)), "| ADHD:", int(np.sum(subj_labels == 1)))


# ============================================================
# PARTICIONES FIJAS
# ============================================================

outer = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED_FOLDS
)

X_dummy = np.zeros((len(subj_names), 1), dtype=np.float32)

FIXED_SPLITS = []
partitions_rows = []

for fold_id, (trainval_subj_idx, test_subj_idx) in enumerate(
    outer.split(X_dummy, subj_labels, groups=subj_names)
):
    print(f"\n================ FIXED FOLD {fold_id + 1}/{N_FOLDS} ================")

    inner = StratifiedGroupKFold(
        n_splits=4,
        shuffle=True,
        random_state=SEED_FOLDS + fold_id
    )

    inner_X_dummy = np.zeros((len(trainval_subj_idx), 1), dtype=np.float32)
    inner_y = subj_labels[trainval_subj_idx]
    inner_groups = subj_names[trainval_subj_idx]

    tr_rel, val_rel = next(
        inner.split(inner_X_dummy, inner_y, groups=inner_groups)
    )

    train_subj_idx = trainval_subj_idx[tr_rel]
    val_subj_idx = trainval_subj_idx[val_rel]

    tr_names = subj_names[train_subj_idx]
    va_names = subj_names[val_subj_idx]
    te_names = subj_names[test_subj_idx]

    tr_labels = subj_labels[train_subj_idx]
    va_labels = subj_labels[val_subj_idx]
    te_labels = subj_labels[test_subj_idx]

    if len(np.unique(tr_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: train set does not contain both classes.")

    if len(np.unique(va_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: validation set does not contain both classes.")

    if len(np.unique(te_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: test set does not contain both classes.")

    tr_idx = epoch_indices_from_subjects(groups, tr_names)
    val_idx = epoch_indices_from_subjects(groups, va_names)
    te_idx = epoch_indices_from_subjects(groups, te_names)

    trainval_names = np.concatenate([tr_names, va_names])
    trainval_idx = epoch_indices_from_subjects(groups, trainval_names)

    FIXED_SPLITS.append({
        "fold_id": fold_id,
        "tr_names": tr_names,
        "va_names": va_names,
        "te_names": te_names,
        "tr_idx": tr_idx,
        "val_idx": val_idx,
        "te_idx": te_idx,
        "trainval_idx": trainval_idx,
    })

    partitions_rows.append({
        "fold": fold_id,
        "train_subjects": list(tr_names),
        "val_subjects": list(va_names),
        "test_subjects": list(te_names),
        "n_train_subjects": len(tr_names),
        "n_val_subjects": len(va_names),
        "n_test_subjects": len(te_names),
        "n_train_epochs": len(tr_idx),
        "n_val_epochs": len(val_idx),
        "n_test_epochs": len(te_idx),
    })

df_partitions = pd.DataFrame(partitions_rows)

df_partitions.to_csv(
    os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"),
    index=False
)

print("\nParticiones fijas guardadas en:")
print(os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"))


# ============================================================
# CORRER UNA SEED
# ============================================================

def run_cnn_lstm_one_seed(training_seed, save_artifacts=True):
    print(f"\n{'=' * 90}")
    print(f"RUN CNN-LSTM EEGNET | training_seed = {training_seed}")
    print(f"{'=' * 90}")

    seed_root = os.path.join(RESULTS_ROOT, f"seed_{training_seed}")
    curves_dir = os.path.join(seed_root, "training_curves")
    models_dir = os.path.join(seed_root, "models")
    cm_dir = os.path.join(seed_root, "cm")
    weights_dir = os.path.join(seed_root, "weights")

    for d in [seed_root, curves_dir, models_dir, cm_dir, weights_dir]:
        os.makedirs(d, exist_ok=True)

    fold_results = []
    fold_histories = {}
    fold_cms = {}
    super_cm = np.zeros((2, 2), dtype=int)

    for split in FIXED_SPLITS:
        fold_id = split["fold_id"]

        print(f"\n---------------- Fold {fold_id + 1}/{N_FOLDS} | seed={training_seed} ----------------")

        fold_seed = int(training_seed * 100 + fold_id)

        trainval_idx = split["trainval_idx"]
        te_idx = split["te_idx"]

        trainval_ds = make_cnn_lstm_ds_from_indices(
            X, y, trainval_idx,
            training=True,
            batch_size=BATCH_SIZE,
            seed=fold_seed
        )

        test_ds_xyg = make_cnn_lstm_ds_from_indices_with_groups(
            X, y, groups, te_idx,
            training=False,
            batch_size=BATCH_SIZE,
            seed=fold_seed
        )

        set_all_seeds(fold_seed)

        model_args = dict(CNN_LSTM_FIXED_MODEL_ARGS)
        model_args["n_channels"] = N_CH
        model_args["n_times"] = N_T

        model = cnn_lstm_eegnet(
            n_channels=model_args["n_channels"],
            n_times=model_args["n_times"],
            n_classes=model_args["n_classes"],
            temporal_filters=model_args["temporal_filters"],
            temporal_kernel=model_args["temporal_kernel"],
            depth_multiplier=model_args["depth_multiplier"],
            pool_size=model_args["pool_size"],
            pool_stride=model_args["pool_stride"],
            dropout_rate=model_args["dropout_rate"],
            lstm_units_1=model_args["lstm_units_1"],
            lstm_units_2=model_args["lstm_units_2"],
            activation=model_args["activation"],
            use_bias_conv=model_args["use_bias_conv"],
            name=model_args["name"],
        )

        opt = Adam(
            learning_rate=CNN_LSTM_FIXED_COMPILE_ARGS["learning_rate"]
        )

        model.compile(
            optimizer=opt,
            loss="sparse_categorical_crossentropy",
            metrics=[
                tf.keras.metrics.SparseCategoricalAccuracy(name="acc"),
            ],
        )

        # ====================================================
        # ENTRENAMIENTO FINAL SOBRE TRAIN + VAL
        # ====================================================

        hist = model.fit(
            trainval_ds,
            epochs=EPOCHS_FINAL,
            verbose=1,
        )

        fold_histories[fold_id] = hist.history

        if save_artifacts:
            model.save(
                os.path.join(models_dir, f"fold_{fold_id}.keras")
            )

            model.save_weights(
                os.path.join(weights_dir, f"fold_{fold_id}.weights.h5")
            )

            plot_fold_training_curve(
                hist.history,
                fold_id,
                save_path=os.path.join(curves_dir, f"fold_{fold_id}_curves.png"),
                metrics=("loss", "acc"),
            )

        # ====================================================
        # EVALUACIÓN A NIVEL DE SUJETO EN TEST
        # ====================================================

        fold_eval = evaluate_subject_level_softmax(
            model,
            test_ds_xyg,
            threshold=0.5
        )

        cm = fold_eval["cm"]
        super_cm += cm
        fold_cms[fold_id] = cm

        if save_artifacts:
            plot_fold_confusion_matrix(
                cm,
                fold_id,
                save_path=os.path.join(cm_dir, f"fold_{fold_id}_cm.png"),
            )

        row = {
            "seed": training_seed,
            "fold": fold_id,
            "accuracy": fold_eval["accuracy"],
            "balanced_acc": fold_eval["balanced_acc"],
            "recall": fold_eval["recall"],
            "precision": fold_eval["precision"],
            "f1": fold_eval["f1"],
            "auc": fold_eval["auc"],
            "model_args": dict(model_args),
            "compile_args": dict(CNN_LSTM_FIXED_COMPILE_ARGS),
        }

        fold_results.append(row)

        print(
            f"Fold {fold_id} | "
            f"Acc={row['accuracy']:.3f} | "
            f"BAcc={row['balanced_acc']:.3f} | "
            f"F1={row['f1']:.3f} | "
            f"AUC={row['auc']:.3f}"
        )

        print("Confusion matrix:\n", cm)

        del model
        gc.collect()
        tf.keras.backend.clear_session()

    if save_artifacts:
        plot_all_training_curves(
            fold_histories,
            curves_dir,
            metrics=("loss", "acc"),
        )

        plot_all_confusion_matrices(
            fold_cms,
            super_cm,
            cm_dir
        )

    df_seed = pd.DataFrame(fold_results)

    seed_summary = {
        "seed": training_seed,
    
        "mean_accuracy": float(df_seed["accuracy"].mean()),
        "std_accuracy": float(df_seed["accuracy"].std(ddof=1)),
    
        "mean_balanced_acc": float(df_seed["balanced_acc"].mean()),
        "std_balanced_acc": float(df_seed["balanced_acc"].std(ddof=1)),
    
        "mean_recall": float(df_seed["recall"].mean()),
        "std_recall": float(df_seed["recall"].std(ddof=1)),
    
        "mean_precision": float(df_seed["precision"].mean()),
        "std_precision": float(df_seed["precision"].std(ddof=1)),
    
        "mean_f1": float(df_seed["f1"].mean()),
        "std_f1": float(df_seed["f1"].std(ddof=1)),
    
        "mean_auc": float(df_seed["auc"].mean()),
        "std_auc": float(df_seed["auc"].std(ddof=1)),
    }

    df_seed.to_csv(
        os.path.join(seed_root, "fold_results.csv"),
        index=False
    )

    pd.DataFrame([seed_summary]).to_csv(
        os.path.join(seed_root, "seed_summary.csv"),
        index=False
    )

    print("\nResumen seed:")
    print(pd.DataFrame([seed_summary]).T)

    return df_seed, seed_summary


# ============================================================
# CORRER LAS 10 SEMILLAS
# ============================================================

all_fold_results = []
all_seed_summaries = []

for training_seed in TRAINING_SEEDS:
    df_seed, seed_summary = run_cnn_lstm_one_seed(
        training_seed=training_seed,
        save_artifacts=True
    )

    all_fold_results.append(df_seed)
    all_seed_summaries.append(seed_summary)

df_all_folds = pd.concat(all_fold_results, axis=0, ignore_index=True)
df_all_seeds = pd.DataFrame(all_seed_summaries)

df_all_folds.to_csv(
    os.path.join(RESULTS_ROOT, "all_fold_results.csv"),
    index=False
)

df_all_seeds.to_csv(
    os.path.join(RESULTS_ROOT, "all_seed_summaries.csv"),
    index=False
)

print("\nRESULTADOS POR SEED")
print(df_all_seeds)

print("\nRESUMEN GLOBAL SOBRE PROMEDIOS POR SEED")
print(
    df_all_seeds[
        [
            "mean_accuracy",
            "mean_balanced_acc",
            "mean_recall",
            "mean_precision",
            "mean_f1",
            "mean_auc",
        ]
    ].agg(["mean", "std"])
)

print("\nRESULTADOS FINALES CNN-LSTM EEGNet - 50 TESTS")

for metric in ["accuracy", "balanced_acc", "recall", "precision", "f1", "auc"]:
    mean_val = df_all_folds[metric].mean() * 100
    std_val = df_all_folds[metric].std(ddof=1) * 100
    print(f"{metric:15s}: {mean_val:.2f} ± {std_val:.2f}")

global_summary_50 = df_all_folds[
    ["accuracy", "balanced_acc", "recall", "precision", "f1", "auc"]
].agg(["mean", "std"])

global_summary_50.to_csv(
    os.path.join(RESULTS_ROOT, "global_summary_50_runs.csv")
)

print("\nArchivos principales guardados en:")
print(" -", os.path.join(RESULTS_ROOT, "all_fold_results.csv"))
print(" -", os.path.join(RESULTS_ROOT, "all_seed_summaries.csv"))
print(" -", os.path.join(RESULTS_ROOT, "global_summary_50_runs.csv"))
print(" -", os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"))

Excluded subjects: ['v56p']
Subjects removed: 1
Remaining subjects: 120
Epoch arrays:
  X: (16640, 19, 256, 1) float32
  y: (16640,) labels: [0 1]
  groups: (16640,) n_subjects: 120
Model input shape (C, T, 1): (19, 256, 1)
Subject table:
  n_subjects: 120
  Control: 59 | ADHD: 61

================ FIXED FOLD 1/5 ================

================ FIXED FOLD 2/5 ================

================ FIXED FOLD 3/5 ================

================ FIXED FOLD 4/5 ================

================ FIXED FOLD 5/5 ================

Particiones fijas guardadas en:
/kaggle/working/results_cnn_lstm_eegnet_10seeds/partitions/fixed_partitions_summary.csv

RUN CNN-LSTM EEGNET | training_seed = 42

---------------- Fold 1/5 | seed=42 ----------------


I0000 00:00:1777562005.790211      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/100


E0000 00:00:1777562013.515292      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1777562013.959587      65 cuda_dnn.cc:529] Loaded cuDNN version 91002


204/204 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - acc: 0.6891 - loss: 0.5816
Epoch 2/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.8184 - loss: 0.4042
Epoch 3/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.8662 - loss: 0.3072
Epoch 4/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.8897 - loss: 0.2638
Epoch 5/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9119 - loss: 0.2125
Epoch 6/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9241 - loss: 0.1837
Epoch 7/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9409 - loss: 0.1525
Epoch 8/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9464 - loss: 0.1378
Epoch 9/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9582 - loss: 0.1019
Epoch 10/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9557 - loss: 0.1121
Epoch 11/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9615 - loss: 0.1030
Epoch 12/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9654 - loss: 0.0919
Epoch 13/100

...

214/214 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - acc: 0.6961 - loss: 0.5823
Epoch 2/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - acc: 0.8065 - loss: 0.4188
Epoch 3/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - acc: 0.8601 - loss: 0.3154
Epoch 4/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - acc: 0.8826 - loss: 0.2794
Epoch 5/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - acc: 0.9008 - loss: 0.2434
Epoch 6/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - acc: 0.9165 - loss: 0.2095
Epoch 7/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9224 - loss: 0.1871
Epoch 8/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9395 - loss: 0.1573
Epoch 9/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - acc: 0.9511 - loss: 0.1328
Epoch 10/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - acc: 0.9510 - loss: 0.1220
Epoch 11/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - acc: 0.9570 - loss: 0.1130
Epoch 12/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9582 - loss: 0.1052
Epoch 13/100


E0000 00:00:1777583372.408246      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


210/210 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - acc: 0.6634 - loss: 0.6069
Epoch 2/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.8056 - loss: 0.4389
Epoch 3/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.8538 - loss: 0.3522
Epoch 4/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.8930 - loss: 0.2670
Epoch 5/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - acc: 0.9113 - loss: 0.2296
Epoch 6/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9296 - loss: 0.1835
Epoch 7/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9429 - loss: 0.1462
Epoch 8/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9476 - loss: 0.1328
Epoch 9/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9544 - loss: 0.1211
Epoch 10/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9489 - loss: 0.1273
Epoch 11/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9616 - loss: 0.1047
Epoch 12/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9606 - loss: 0.1059
Epoch 13/100


E0000 00:00:1777583826.798318      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


212/212 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - acc: 0.6482 - loss: 0.6301
Epoch 2/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - acc: 0.7817 - loss: 0.4688
Epoch 3/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - acc: 0.8362 - loss: 0.3755
Epoch 4/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - acc: 0.8571 - loss: 0.3315
Epoch 5/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - acc: 0.8869 - loss: 0.2690
Epoch 6/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - acc: 0.9099 - loss: 0.2156
Epoch 7/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9193 - loss: 0.1980
Epoch 8/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9344 - loss: 0.1699
Epoch 9/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9395 - loss: 0.1558
Epoch 10/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9483 - loss: 0.1364
Epoch 11/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - acc: 0.9469 - loss: 0.1369
Epoch 12/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.9553 - loss: 0.1210
Epoch 13/100
